In [3]:
import sys
!{sys.executable} -m pip install librosa

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 3.7 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 9.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.7/206.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.1/31.1 MB 9.4 MB/s eta 0:00:0000:01:00:01
  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Hom

In [4]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import IPython.display as ipd

In [5]:
DATA_ROOT = Path("../data/mimii/fan")
SAMPLE_RATE = 16_000

In [ ]:
# --- 2. Load one normal and one abnormal file ---
normal_files  = sorted((DATA_ROOT / "test/normal").glob("*.wav"))
abnormal_files = sorted((DATA_ROOT / "test/abnormal").glob("*wav"))

print(f"Normal files: {len(normal_files)}")
print(f"Abnormal files: {len(abnormal_files)}")

y_normal, sr = librosa.load(normal_files[0], sr=SAMPLE_RATE)
y_abnormal, _ = librosa.load(abnormal_files[0], sr=SAMPLE_RATE)

print(f"\nSample rate: {sr} HZ")
print(f"Duration: {len(y_normal)/sr:.1f} s")
print(f"Samples: {len(y_normal)}")

In [ ]:
# --- 3. Listen (works in Colab/Jupyter) ---
print("\n> Normal:")
ipd.display(ipd.Audio(y_normal, rate=sr))

print("> Abnormal:")
ipd.display(ipd.Audio(y_abnormal, rate=sr))

In [9]:
# --- 4. Waveform comparison --- 
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

librosa.display.waveshow(y_normal, sr=sr, ax=axes[0], color="steelblue")
axes[0].set_title("Waveform - NORMAL")
axes[0].set_ylabel("Amplitude")

librosa.display.waveshow(y_normal, sr=sr, ax=axes[1], color="crimson")
axes[1].set_title("Waveform - ABNORMAL")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.savefig("waveform_comparison.png", dpi=150)
plt.show()

In [10]:
# --- 5. Mel-Spectogram comparison ---
def plot_mel(y, sr, ax, title, color_map="magma"):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=sr, x_axis="time", 
                                   y_axis="mel", fmax=8000, 
                                   ax=ax, cmap=color_map)
    ax.set_title(title)
    return img

fig, axis = plt.subplos(1, 2, figsize=(14, 4))
plot_mel(y_normal, sr, axes[0], "Mel-Spectrogram - NORMAL")
plot_mel(y_abnormal, sr, axes[0], "Mel-Spectrogram - ABNORMAL")
plt.tight_layout()
plt.savefig("mel_comparison.png", dpi=150)
plt.show()

In [ ]:
# --- 6. Basic statistics ---
print("\n Signal statistics")
for label, y in [("Normal  ", y_normal), ("Abnormal", y_abnormal)]:
    rms = np.sqrt(np.mean(y**2))
    print(f"{label} | RMS: {rms:.4f} | MAX: {np.max(np.abs(y)):.4f}"
         f" | Std: {np.std(y):.4f}")